# rtdetr-sportsmot — sports player detection, ready to run

[`smallTech/rtdetr-sportsmot`](https://huggingface.co/smallTech/rtdetr-sportsmot)
is an RT-DETRv2 (r50vd) detector fine-tuned on
[SportsMOT](https://huggingface.co/datasets/Lekim89/sportsmot) to detect
**players** (single class) in sports broadcast footage — built as the
detection stage of a tracking-by-detection pipeline.

Benchmarks (details on the [model card](https://huggingface.co/smallTech/rtdetr-sportsmot)):
**mAP@50 0.938** on 45 unseen val sequences · **28 fps** batched-fp16
throughput / ~62 ms single-image latency on a T4.

This notebook runs on Colab or Kaggle, CPU or GPU (a few seconds per image on
CPU, faster with any GPU): install → load → detect → visualize.

In [ ]:
# --- 1. Dependencies ---------------------------------------------------------
# transformers provides RT-DETRv2; pillow draws the boxes. The checkpoint
# stores every weight explicitly (no tied-weight surprises), so any
# reasonably recent transformers works.
%pip install -q "transformers>=4.48" pillow requests torch torchvision

In [ ]:
# --- 2. Load the model from the Hub ------------------------------------------
import torch
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "smallTech/rtdetr-sportsmot"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"loaded {MODEL_ID} on {DEVICE} "
      f"({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params)")

In [ ]:
# --- 3. Get an image ----------------------------------------------------------
# A real frame from the SportsMOT val split (public). Swap in your own image:
#   image = Image.open("my_frame.jpg").convert("RGB")
import requests
from io import BytesIO
from PIL import Image

FRAME_URL = ("https://huggingface.co/datasets/Lekim89/sportsmot/resolve/main/"
             "val/v_00HRwkvvjtQ_c001/img1/000001.jpg")
image = Image.open(BytesIO(requests.get(FRAME_URL, timeout=60).content)).convert("RGB")
image

In [ ]:
# --- 4. Detect players --------------------------------------------------------
CONF = 0.5                       # confidence threshold — lower to find more players

with torch.no_grad():
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    (result,) = processor.post_process_object_detection(
        outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=CONF)

print(f"{len(result['scores'])} players detected @ conf >= {CONF}")
for score, box in zip(result["scores"], result["boxes"]):
    x1, y1, x2, y2 = (round(float(v)) for v in box)
    print(f"  conf {float(score):.3f}  box ({x1}, {y1}) -> ({x2}, {y2})")

In [ ]:
# --- 5. Visualize -------------------------------------------------------------
from PIL import ImageDraw

annotated = image.copy()
draw = ImageDraw.Draw(annotated)
for score, box in zip(result["scores"], result["boxes"]):
    x1, y1, x2, y2 = (float(v) for v in box)
    draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 100), width=3)
    draw.text((x1 + 3, y1 + 2), f"{float(score):.2f}", fill=(0, 255, 100))
annotated

## Where to go from here

- **Video tracking** — feed per-frame detections into
  [ByteTrack via the `supervision` library](https://supervision.roboflow.com/)
  to get persistent player IDs across a clip; this detector was built exactly
  for that role.
- **Speed** — on a GPU, batch several frames per forward pass and wrap the
  forward in `torch.autocast("cuda", dtype=torch.float16)`: on a T4 that
  takes throughput from ~16 fps (single image) to ~28 fps.
- **How it was trained and benchmarked** — the full Kaggle pipeline (staging,
  training, evaluation, testing) lives in the
  [workspace repository](https://github.com/Tommykewl/HuggingFace).